[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C12_Responsible_AI_Course/01_bias_fairness/01_bias_fairness.ipynb)

# 01 · 偏见与公平性度量（从零实现）

目标：用 numpy/pandas 把 **四种群体公平定义**（demographic parity / equalized odds / predictive parity / calibration）从零算出来，并**亲手演示不可能定理**——构造一个完美校准的分数，看它在基率不同的两组上如何被迫产生不等的假阳率。

路线：子群混淆矩阵 → demographic parity & 80% 规则 → equalized odds → predictive parity & 校准 → **不可能定理冲突** → ✏️ 练习 → 📖 答案 → 🧪 COMPAS 真实数据胶囊。

> 心智模型：**每个公平定义都在要求某个率跨组相等；不可能定理说基率不同时，校准/TPR均等/FPR均等这三个不能全都要。**

## 1 · 子群混淆矩阵：一切的起点

先把「按组分别算 TPR/FPR/PPV/选中率」写出来。这是后面每个公平定义的公共底座。

我们造一个带受保护属性 `group` 的玩具招聘数据：标签 `y`=真实合格，`pred`=模型决策。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

def rates(y_true, y_pred):
    '''从混淆矩阵派生 TPR/FPR/PPV/选中率(sel)/基率。分母为 0 返回 nan。'''
    y_true = np.asarray(y_true).astype(int); y_pred = np.asarray(y_pred).astype(int)
    TP = np.sum((y_pred==1)&(y_true==1)); FP = np.sum((y_pred==1)&(y_true==0))
    FN = np.sum((y_pred==0)&(y_true==1)); TN = np.sum((y_pred==0)&(y_true==0))
    def s(a,b): return a/b if b>0 else float('nan')
    return dict(TPR=s(TP,TP+FN), FPR=s(FP,FP+TN), PPV=s(TP,TP+FP),
                sel=s(TP+FP, TP+FP+FN+TN), base=s(TP+FN, TP+FP+FN+TN), n=int(TP+FP+FN+TN))

def subgroup_table(df, group_col='group', ycol='y', pcol='pred'):
    return pd.DataFrame({g: rates(s[ycol], s[pcol]) for g, s in df.groupby(group_col)}).T

# 两组：A、B 各 400 人。两组真实基率相同(0.5)，但模型对 A 组假阳更多
def make_group(n, group, fp_rate, fn_rate, base=0.5):
    y = (rng.random(n) < base).astype(int)
    pred = y.copy()
    pred[(y==0)&(rng.random(n)<fp_rate)] = 1   # 假阳
    pred[(y==1)&(rng.random(n)<fn_rate)] = 0   # 假阴
    return pd.DataFrame(dict(group=group, y=y, pred=pred))

df = pd.concat([make_group(400,'A',fp_rate=0.30,fn_rate=0.10),
                make_group(400,'B',fp_rate=0.08,fn_rate=0.10)], ignore_index=True)
tab = subgroup_table(df)
print(tab[['base','sel','TPR','FPR','PPV','n']].round(3))
assert tab.loc['A','FPR'] > tab.loc['B','FPR'] + 0.1, 'A 组假阳率应明显更高'
print('\n✅ 子群混淆矩阵就位：A 组 FPR 远高于 B —— 整体指标会掩盖它')

## 2 · demographic parity 与 80% 规则

demographic parity 要求各组**选中率相等**（不看标签）。其连续度量是 **disparate impact ratio** = 弱势组选中率 / 优势组选中率，<0.8 触发 80% 规则预警。

下面造一个选中率不均的场景并算这两个量。

In [ ]:
def demographic_parity_diff(df, group_col='group', pcol='pred'):
    '''最大组间选中率之差。0 = 完美人口均等。'''
    sel = df.groupby(group_col)[pcol].mean()
    return float(sel.max() - sel.min()), sel

def disparate_impact_ratio(df, advantaged, disadvantaged, group_col='group', pcol='pred'):
    sel = df.groupby(group_col)[pcol].mean()
    return float(sel[disadvantaged] / sel[advantaged])

hire = pd.concat([
    pd.DataFrame(dict(group='M', pred=(rng.random(500)<0.50).astype(int))),
    pd.DataFrame(dict(group='F', pred=(rng.random(500)<0.28).astype(int))),
], ignore_index=True)
dp, sel = demographic_parity_diff(hire)
di = disparate_impact_ratio(hire, advantaged='M', disadvantaged='F')
print('各组选中率:'); print(sel.round(3))
print(f'demographic parity 差距 = {dp:.3f}')
print(f'disparate impact 比 (F/M) = {di:.2f}  -> ' + ('预警 ⚠️' if di<0.8 else 'OK'))
assert dp > 0.15 and di < 0.8
print('✅ 选中率差距大、DI<0.8：违反人口均等并触发 80% 规则')

## 3 · equalized odds：TPR 差距 + FPR 差距

equalized odds 要求 **TPR 和 FPR 都跨组相等**。度量它就是分别看两个差距，取较大者（或都报）。

用第 1 节的 `df`（A 组假阳更多），它的 TPR 差距小、FPR 差距大 —— 典型的「机会均等达标但 equalized odds 不达标」。

In [ ]:
def equalized_odds_gaps(df, group_col='group', ycol='y', pcol='pred'):
    '''返回 (TPR 差距, FPR 差距, 取大者)。'''
    tab = subgroup_table(df, group_col, ycol, pcol)
    tpr_gap = float(tab['TPR'].max() - tab['TPR'].min())
    fpr_gap = float(tab['FPR'].max() - tab['FPR'].min())
    return tpr_gap, fpr_gap, max(tpr_gap, fpr_gap)

tpr_gap, fpr_gap, eo = equalized_odds_gaps(df)
print(f'TPR 差距 = {tpr_gap:.3f}   (equal opportunity 看这个)')
print(f'FPR 差距 = {fpr_gap:.3f}   (这一项让 equalized odds 不达标)')
print(f'equalized odds 违反程度 = {eo:.3f}')
assert fpr_gap > tpr_gap, '本例 FPR 差距应远大于 TPR 差距'
assert tpr_gap < 0.1, 'TPR 差距小 -> equal opportunity 近似达标'
print('✅ 机会均等(TPR)近似达标，但 FPR 差距使 equalized odds 失败 —— 两个定义会给出不同结论')

## 4 · 组内校准：分数对每个组意味同一件事吗

校准要求：在每个分数桶里，真实正类比例 ≈ 该分数。把分数分箱，按组算每箱的真实正类率，与箱中心比。

我们先造一个**完美校准**的分数（标签直接由分数当概率采样），验证校准误差≈0。

In [ ]:
def calibration_by_group(df, group_col='group', scol='score', ycol='y', n_bins=10):
    '''返回每组的 ECE(期望校准误差) 与分箱明细。ECE=Σ_bin (n_bin/n)·|实际正类率 − 平均分数|。'''
    out = {}
    bins = np.linspace(0, 1, n_bins+1)
    for g, s in df.groupby(group_col):
        sc = s[scol].values; y = s[ycol].values
        idx = np.clip(np.digitize(sc, bins) - 1, 0, n_bins-1)
        ece = 0.0
        for b in range(n_bins):
            m = idx==b
            if m.sum()==0: continue
            conf = sc[m].mean(); acc = y[m].mean()
            ece += (m.sum()/len(sc)) * abs(acc - conf)
        out[g] = ece
    return out

# 完美校准：score~U(0,1)，y ~ Bernoulli(score)，两组基率不同(通过分数分布不同)
def make_calibrated(n, group, score_mean):
    score = np.clip(rng.normal(score_mean, 0.2, n), 0.01, 0.99)
    y = (rng.random(n) < score).astype(int)   # 标签由分数当真实概率生成 -> 必然校准
    return pd.DataFrame(dict(group=group, score=score, y=y))

cal = pd.concat([make_calibrated(4000,'A',0.60), make_calibrated(4000,'B',0.40)], ignore_index=True)
ece = calibration_by_group(cal)
print('各组 ECE(越小越校准):', {g: round(v,3) for g,v in ece.items()})
print('各组真实基率:', cal.groupby('group')['y'].mean().round(3).to_dict())
assert max(ece.values()) < 0.05, '构造上两组都应近似完美校准'
print('✅ 两组都校准良好，但真实基率不同(A≈0.6, B≈0.4) —— 这正是不可能定理的火药')

## 5 · 亲手撞上不可能定理 🔥

上面那个**完美校准**、但**两组基率不同**的分数 `cal`，现在用**同一个阈值**把它变成 0/1 决策。

**断言**：因为校准成立 + 基率不同，两组的 **FPR 必然不相等**。我们让代码证明给自己看 —— 这就是 Kleinberg/Chouldechova。

In [ ]:
def threshold_predict(df, thr, scol='score'):
    out = df.copy(); out['pred'] = (out[scol].values >= thr).astype(int); return out

dec = threshold_predict(cal, thr=0.5)
tab = subgroup_table(dec)
print(tab[['base','TPR','FPR','PPV']].round(3))
fpr_gap = abs(tab.loc['A','FPR'] - tab.loc['B','FPR'])
tpr_gap = abs(tab.loc['A','TPR'] - tab.loc['B','TPR'])
ppv_gap = abs(tab.loc['A','PPV'] - tab.loc['B','PPV'])
print(f'\nFPR 差距={fpr_gap:.3f}  TPR 差距={tpr_gap:.3f}  PPV 差距={ppv_gap:.3f}')
# 校准成立 + 基率不同 => 不可能同时让 FPR 和 TPR 都相等
assert fpr_gap > 0.05, '基率高的 A 组必然有更高 FPR —— 这是定理，不是 bug'
print('\n🔥 一个完美校准的分数，在基率不同的两组上，被同一阈值逼出了不等的 FPR。')
print('   想消除 FPR 差距？只能放弃校准或放弃 TPR 相等 —— 三者锁死。')

**验证那条锁死的恒等式** $\mathrm{FPR}=\frac{b}{1-b}\cdot\frac{1-p}{p}\cdot\mathrm{TPR}$（Chouldechova）。我们用 A 组的 base/PPV/TPR 反推 FPR，看是否与直接算的一致。

In [ ]:
def fpr_from_identity(base, ppv, tpr):
    '''Chouldechova 恒等式：FPR = base/(1-base) * (1-ppv)/ppv * TPR'''
    return base/(1-base) * (1-ppv)/ppv * tpr

for g in ['A','B']:
    r = tab.loc[g]
    pred_fpr = fpr_from_identity(r['base'], r['PPV'], r['TPR'])
    print(f'{g}: 直接算 FPR={r["FPR"]:.3f}  恒等式推 FPR={pred_fpr:.3f}')
    assert abs(pred_fpr - r['FPR']) < 1e-6, '恒等式必须精确成立'
print('✅ 恒等式精确成立：FPR 被 base/PPV/TPR 锁死 —— 基率一变，FPR 跟着变')

---
## ✏️ 练习 1：demographic parity 差距

实现 `dp_diff(df, group_col, pcol)`：返回**最大组间选中率之差**（选中率 = 预测为 1 的比例）。0 表示完美人口均等。

In [ ]:
def dp_diff(df, group_col='group', pcol='pred'):
    # TODO: 按 group_col 分组算选中率(pred 均值)，返回 max - min
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ex = pd.concat([pd.DataFrame(dict(group='X', pred=(rng.random(300)<0.6).astype(int))),
                pd.DataFrame(dict(group='Y', pred=(rng.random(300)<0.35).astype(int)))], ignore_index=True)
d = dp_diff(ex)
assert 0.15 < d < 0.35, f'X≈0.6,Y≈0.35 -> 差距约 0.25, got {d:.3f}'
# 同组复制应得 0 差距
same = pd.DataFrame(dict(group=['X']*100+['Y']*100, pred=[1,0]*100))
assert abs(dp_diff(same)) < 1e-9
print(f'✅ 练习 1 通过：DP 差距 = {d:.3f}')

## ✏️ 练习 2：equalized odds 差距

实现 `eo_gaps(df, ...)`：返回 `(TPR 差距, FPR 差距)`。复用上面的 `rates`。

提示：对每组算 `rates`，取 TPR 的 max-min、FPR 的 max-min。

In [ ]:
def eo_gaps(df, group_col='group', ycol='y', pcol='pred'):
    # TODO: 每组 rates()，返回 (TPR 的 max-min, FPR 的 max-min)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
tg, fg = eo_gaps(df)   # 第 1 节的 df：A 组假阳多
assert fg > 0.15, 'FPR 差距应该大'
assert tg < 0.1, 'TPR 差距应该小'
# 完全公平的数据应得 ~0
fair = pd.concat([make_group(500,'A',0.1,0.1), make_group(500,'B',0.1,0.1)], ignore_index=True)
tg2, fg2 = eo_gaps(fair)
assert tg2 < 0.1 and fg2 < 0.1, '同分布两组差距应小'
print(f'✅ 练习 2 通过：TPR 差距={tg:.3f}, FPR 差距={fg:.3f}')

## ✏️ 练习 3：组内校准误差（ECE）

实现 `group_ece(df, group_col, scol, ycol, n_bins)`：对每组把分数分箱，返回每组的 ECE = $\sum_\text{bin} \frac{n_\text{bin}}{n}\,|\text{该箱真实正类率} - \text{该箱平均分数}|$。返回 dict{组: ECE}。

In [ ]:
def group_ece(df, group_col='group', scol='score', ycol='y', n_bins=10):
    # TODO: 对每组，用 np.digitize 把 score 分到 n_bins 个箱，
    #       每箱算 |真实正类率 - 平均分数| 并按箱内样本占比加权求和
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
e = group_ece(cal)   # 第 4 节的完美校准数据
assert max(e.values()) < 0.05, '完美校准数据 ECE 应接近 0'
# 故意破坏校准：把分数整体抬高 0.3 但标签不变 -> ECE 变大
bad = cal.copy(); bad['score'] = np.clip(bad['score'] + 0.3, 0, 1)
eb = group_ece(bad)
assert min(eb.values()) > max(e.values()), '破坏校准后 ECE 应明显增大'
print(f'✅ 练习 3 通过：校准数据 ECE={ {g:round(v,3) for g,v in e.items()} }, 破坏后={ {g:round(v,3) for g,v in eb.items()} }')

## ✏️ 练习 4：为 equal opportunity 找每组阈值

后处理：给定每组的分数，找**每组各自的阈值**使两组 **TPR 都达到目标 `target_tpr`**（equal opportunity）。

实现 `threshold_for_tpr(scores, y, target_tpr)`：返回使该组 TPR ≥ target 的**最高**阈值（最高阈值 = 在达标前提下尽量少误抓）。

In [ ]:
def threshold_for_tpr(scores, y, target_tpr):
    # TODO: 在候选阈值(如 np.unique(scores))里，找使 TPR>=target_tpr 的最大阈值。
    #       TPR = (正类中 score>=thr 的比例)。若无满足则返回最小分数(全选中)。
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
gA = cal[cal['group']=='A']; gB = cal[cal['group']=='B']
thrA = threshold_for_tpr(gA['score'].values, gA['y'].values, 0.8)
thrB = threshold_for_tpr(gB['score'].values, gB['y'].values, 0.8)
def tpr_at(s, y, thr): 
    pos = y==1; return ((s[pos]>=thr).mean())
tA = tpr_at(gA['score'].values, gA['y'].values, thrA)
tB = tpr_at(gB['score'].values, gB['y'].values, thrB)
print(f'A 阈值={thrA:.3f} -> TPR={tA:.3f}   B 阈值={thrB:.3f} -> TPR={tB:.3f}')
assert tA >= 0.8 - 1e-9 and tB >= 0.8 - 1e-9, '两组 TPR 都应达标'
assert abs(tA - tB) < 0.08, 'equal opportunity: 两组 TPR 应接近'
# 两组阈值一般不同 —— 这正是「按组用不同阈值」的后处理(有法律争议)
print('✅ 练习 4 通过：为达到 equal opportunity，两组用了不同阈值')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def dp_diff(df, group_col='group', pcol='pred'):
    sel = df.groupby(group_col)[pcol].mean()
    return float(sel.max() - sel.min())

In [ ]:
# 练习 2 参考答案
def eo_gaps(df, group_col='group', ycol='y', pcol='pred'):
    tab = subgroup_table(df, group_col, ycol, pcol)
    return float(tab['TPR'].max()-tab['TPR'].min()), float(tab['FPR'].max()-tab['FPR'].min())

In [ ]:
# 练习 3 参考答案
def group_ece(df, group_col='group', scol='score', ycol='y', n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    out = {}
    for g, s in df.groupby(group_col):
        sc = s[scol].values; y = s[ycol].values
        idx = np.clip(np.digitize(sc, bins)-1, 0, n_bins-1)
        ece = 0.0
        for b in range(n_bins):
            m = idx==b
            if m.sum()==0: continue
            ece += (m.sum()/len(sc)) * abs(y[m].mean() - sc[m].mean())
        out[g] = float(ece)
    return out

In [ ]:
# 练习 4 参考答案
def threshold_for_tpr(scores, y, target_tpr):
    scores = np.asarray(scores); y = np.asarray(y)
    pos = scores[y==1]
    best = scores.min()
    for thr in np.unique(scores):
        tpr = (pos >= thr).mean()
        if tpr >= target_tpr and thr > best:
            best = thr
    return float(best)

---
## 🧪 真实数据胶囊：COMPAS —— 不可能定理的真实现场

**COMPAS** 是美国再犯风险评估工具，引发公平争议的经典案例。我们尝试联网下载 ProPublica 的 COMPAS 数据，按种族（African-American vs Caucasian）看：以 `decile_score≥5` 为「高危」决策，两组的 **FPR 差距**（ProPublica 的指控）与各组**再犯基率**。**下载失败自动回退到内置真实统计数值**（取自 ProPublica 分析的已知结果），逻辑不变。

In [ ]:
import io, urllib.request

COMPAS_URL = 'https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv'

def load_compas():
    '''返回 (df[race,decile_score,two_year_recid], source)。失败回退内置真实统计。'''
    try:
        req = urllib.request.Request(COMPAS_URL, headers={'User-Agent':'Mozilla/5.0'})
        raw = urllib.request.urlopen(req, timeout=8).read().decode('utf-8','replace')
        d = pd.read_csv(io.StringIO(raw))
        # ProPublica 的标准过滤
        d = d[(d['days_b_screening_arrest']<=30)&(d['days_b_screening_arrest']>=-30)&
              (d['is_recid']!=-1)&(d['c_charge_degree']!='O')&(d['score_text']!='N/A')]
        d = d[d['race'].isin(['African-American','Caucasian'])]
        d = d[['race','decile_score','two_year_recid']].dropna()
        return d, 'COMPAS (ProPublica, downloaded)'
    except Exception as e:
        print('下载失败, 回退内置真实统计:', type(e).__name__)
        # 内置：基率 AA≈0.51, C≈0.39；decile 分布按真实近似(AA 偏高分)
        rngc = np.random.default_rng(7)
        def synth(n, race, base, hi_score_rate):
            recid = (rngc.random(n) < base).astype(int)
            # 高危者更可能拿高分；用 base 调制 decile 大致校准
            p_hi = np.where(recid==1, hi_score_rate+0.15, hi_score_rate-0.10)
            decile = np.where(rngc.random(n) < np.clip(p_hi,0.05,0.95),
                              rngc.integers(5,11,n), rngc.integers(1,5,n))
            return pd.DataFrame(dict(race=race, decile_score=decile, two_year_recid=recid))
        d = pd.concat([synth(3000,'African-American',0.51,0.55),
                       synth(2400,'Caucasian',0.39,0.42)], ignore_index=True)
        return d, 'built-in (synthetic from real COMPAS rates)'

compas, src = load_compas()
print('数据来源:', src, '| 样本数:', len(compas))
compas['high_risk'] = (compas['decile_score'] >= 5).astype(int)   # 决策：高危
summary = compas.groupby('race').apply(
    lambda s: pd.Series(rates(s['two_year_recid'], s['high_risk'])), include_groups=False)
print(summary[['base','FPR','TPR','PPV']].round(3))
aa, ca = 'African-American', 'Caucasian'
fpr_gap = summary.loc[aa,'FPR'] - summary.loc[ca,'FPR']
print(f'\n再犯基率: AA={summary.loc[aa,"base"]:.2f}, C={summary.loc[ca,"base"]:.2f}')
print(f'FPR(被误标高危) 差距 AA-C = {fpr_gap:.3f}')
assert summary.loc[aa,'base'] > summary.loc[ca,'base'], '真实数据中 AA 再犯基率更高'
assert fpr_gap > 0.05, 'ProPublica 的发现：AA 的 FPR 更高'
print('✅ 真实重现：基率较高的群体被同一分数阈值逼出更高 FPR —— 不可能定理的现实写照')

**🧪 胶囊练习**：实现 `which_fairness_violated(summary, race_a, race_b, tol=0.05)`：输入上面的 `summary` 表，返回一个 dict 标明 `demographic_parity`/`equal_opportunity`/`fpr_balance` 三者**各是否被违反**（对应 sel / TPR / FPR 的组间差距是否 > tol）。

In [ ]:
def which_fairness_violated(summary, race_a, race_b, tol=0.05):
    # TODO: 比较 summary.loc[race_a] 与 summary.loc[race_b] 的 sel/TPR/FPR,
    #       返回 {'demographic_parity':bool, 'equal_opportunity':bool, 'fpr_balance':bool}
    #       True 表示该准则被违反(差距 > tol)
    raise NotImplementedError

In [ ]:
# 自测
v = which_fairness_violated(summary, aa, ca)
assert v['fpr_balance'] is True, 'FPR 差距大 -> fpr_balance 被违反'
assert set(v.keys()) == {'demographic_parity','equal_opportunity','fpr_balance'}
print('违反情况:', v)
print('✅ 胶囊练习通过：用代码标定 COMPAS 违反了哪几条公平准则')

In [ ]:
# 📖 胶囊参考答案
def which_fairness_violated(summary, race_a, race_b, tol=0.05):
    a = summary.loc[race_a]; b = summary.loc[race_b]
    return {
        'demographic_parity': bool(abs(a['sel'] - b['sel']) > tol),
        'equal_opportunity':  bool(abs(a['TPR'] - b['TPR']) > tol),
        'fpr_balance':        bool(abs(a['FPR'] - b['FPR']) > tol),
    }

### 小结
- 每个群体公平定义都在要求**某个率跨组相等**：sel(DP) / TPR(equal opp) / TPR+FPR(equalized odds) / PPV(predictive parity) / 校准。
- 它们归为三族：independence / separation / sufficiency，**一般不能同时满足**。
- **不可能定理**：基率不同 + 不完美分类器 ⇒ 校准、TPR 均等、FPR 均等三者最多取二。恒等式 $\mathrm{FPR}=\frac{b}{1-b}\frac{1-p}{p}\mathrm{TPR}$ 把它们锁死。
- **COMPAS** 是活教材：双方都对，争的是该优先哪种公平 —— 这是价值判断，不是统计能裁决的。

下一站：**模块 02 · 毒性与有害内容评测** —— 把子群切片从「受保护属性」换成「身份词」，审计内容审核的意外偏差。